# TICKET MJD-502

De: Elena Pérez · Responsable de datos
Tenemos la misma regla de normalización del SKU en dos sitios: en SQL (silver) y en Python
( marejada_utils.py ). Quiero estar segura de que hacen lo mismo. Pruébala con los casos raros y com‐
párala con silver fila a fila.

In [ ]:
import marejada_utils as mu

casos = {
    "CER-0001": "CER-0001",   # ya correcto: no debe cambiar
    "cer-0012": "CER-0012",   # minúsculas
    " CER-0012": "CER-0012",  # espacio delante
    "CER0007 ": "CER-0007",   # sin guion y espacio detrás
    "obs-0003": "OBS-0003",   # referencia antigua: también se normaliza
}
for entrada, esperado in casos.items():
    obtenido = mu.normalizar_sku(entrada)
    assert obtenido == esperado, f"{entrada!r}: esperaba {esperado!r} y salió {obtenido!r}"
print(f"{len(casos)} casos correctos")


Límite conocido de la regla: un SKU con ceros de menos (`CER-12`) **no** se corrige; devuelve `CER-12`, no `CER-0012`. Ningún SKU del inventario está escrito así.

Comparación con silver, fila a fila:

In [ ]:
import duckdb

con = duckdb.connect("../../db/marejada.duckdb", read_only=True)
silver = con.sql("""
    SELECT b.almacen, b.sku AS sku_original, b.ubicacion, s.sku AS sku_silver
    FROM bronze.inventario AS b
    JOIN silver.inventario AS s
      ON s.almacen = b.almacen AND s.ubicacion = b.ubicacion AND s.descripcion = b.descripcion
""").fetchall()
con.close()

distintos = [(a, orig, sil) for a, orig, ubi, sil in silver if mu.normalizar_sku(orig) != sil]
print(f"Filas comparadas: {len(silver)}")
print(f"Filas en las que Python y SQL no coinciden: {len(distintos)}")
